# EMG 與腳踏車力矩整合分析 — 中位數（Median）版

這份 Notebook 由原始文字程式整理而成，主要流程如下：

1. 讀取腳踏車力矩（Torque）Excel 與 EMG Excel。
2. 分別去除兩份資料前後各 60 秒。
3. 將左右腳力矩依曲柄角度 0–359° 做中位數（Median）統計。
4. 自動尋找左右腳力矩交叉點，校正角度 Offset。
5. 以圓形移動平均降低力矩曲線雜訊。
6. 從左右 Vastus Medialis（股內側肌）EMG 的交叉點切分踩踏週期。
7. 將每一個 EMG 週期重新取樣到 0–359°。
8. 計算跨週期 EMG 的中位數（Median）曲線。
9. 匯出 Torque、EMG 與 cycle timing 到 Excel。
10. 繪製左右腳 Torque 與 VM EMG 比較圖。

> 注意：EMG 的 0–359° 是依「單一週期正規化」換算而來，不是直接量測到的實際曲柄角度。


## 執行前準備

請先把以下兩個 Excel 放在 Notebook 同一個資料夾：

- `C.xlsx`：腳踏車曲柄角度與左右力矩資料
- `C EMG.xlsx`：EMG 資料

若檔名不同，請到「USER CONFIGURATION」區修改 `CPM_FILE` 與 `EMG_FILE`。

需要套件：
- numpy
- pandas
- matplotlib
- openpyxl

一般 Anaconda / Jupyter 通常只需要執行下一格安裝指令一次。


In [ ]:
# 第一次執行時若缺少套件，取消下一行註解後執行一次
# %pip install numpy pandas matplotlib openpyxl


In [ ]:
﻿# 分別去除力矩與 EMG 原始記錄前後各 60 秒，再進行所有分析。
# 保留原始檔案與時間戳；保留原程式交叉點對齊、重採樣及移動平均平滑。
# 無效 EMG 週期以 NaN 排除，避免零值參與統計。
# 各自裁切不代表時間同步；原程式的 EMG 角度標示為週期換算，非實測角度。
# 本檔保留原筆記本 piplite/%pip 語法，使用方式同原始文字檔。

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime


## 1. 參數設定

這裡設定輸入檔名、工作表、裁切秒數、欄位名稱與平滑參數。

重點：
- `TRIM_START_SECONDS` / `TRIM_END_SECONDS`：前後各刪除幾秒。
- `TIME_TQ`：Torque 時間欄位。
- `ANGLE_TQ`：曲柄角度欄位。
- `TORQUE_L` / `TORQUE_R`：左右腳力矩欄位。
- `R_VM_COL` / `L_VM_COL`：左右股內側肌 EMG 欄位。
- `TORQUE_SMOOTH_WINDOW_DEG`：力矩曲線平滑視窗。


In [ ]:
# ---- CPM torque file ----
CPM_FILE  = "C.xlsx"   # ← change if needed
CPM_SHEET = 0

TRIM_START_SECONDS = 60.0
TRIM_END_SECONDS = 60.0

TIME_TQ   = "Timestamp (s)"
ANGLE_TQ  = "Angle"
TORQUE_L  = "crankLeft"
TORQUE_R  = "CrankRight"

# If angle is "late", make it earlier by using a negative offset, e.g. -25
ANGLE_OFFSET_DEG = 35 # ← adjust after you see the plots


# ---- EMG file ----
EMG_FILE = "C EMG.xlsx"  # ← change if needed

EMG_SHEET = 0

TIME_EMG  = "X [s]"
R_VM_COL  = "R VASTUS MEDIALIS: EMG 3->Filter->RESAMP->RMS->Amplitude Analysis [Percent]"
L_VM_COL  = "L VASTUS MEDIALIS: EMG 4->Filter->RESAMP->RMS->Amplitude Analysis [Percent]"

# ---- SMOOTHING / NOISE FILTER FOR MEDIAN TORQUE ----
APPLY_TORQUE_SMOOTHING = True   # set False if you don't want smoothing
TORQUE_SMOOTH_WINDOW_DEG = 13    # moving-average window in degrees (odd number recommended)


## 2. 共用函式

主要有兩個功能：

- `trim_edges()`：依每份資料自己的時間範圍，刪除前後指定秒數。
- `median_vs_angle()`：把 0–359° 每一度內的資料做中位數（Median）統計。
- `moving_average_circular()`：對 360° 週期資料做圓形移動平均，避免 0° / 360° 邊界被切斷。


In [ ]:
def trim_edges(df, time_col, label):
    """依每份檔案自己的時間範圍裁切，保留原時間戳，不做同步。"""
    df = df.copy()
    df[time_col] = pd.to_numeric(df[time_col], errors="raise")
    if len(df) < 2 or not np.isfinite(df[time_col].to_numpy()).all():
        raise ValueError(f"{label}：時間欄位無效或資料不足。")
    if TRIM_START_SECONDS < 0 or TRIM_END_SECONDS < 0:
        raise ValueError("裁切秒數不可為負數。")
    df = df.sort_values(time_col)
    start = df[time_col].iloc[0] + TRIM_START_SECONDS
    end = df[time_col].iloc[-1] - TRIM_END_SECONDS
    if start >= end:
        raise ValueError(f"{label}：資料長度不足以裁切前後指定秒數。")
    result = df.loc[df[time_col].between(start, end)].copy()
    if len(result) < 2:
        raise ValueError(f"{label}：裁切後資料不足。")
    print(f"{label}：保留時間 {start:.3f}～{end:.3f} 秒，{len(df)} → {len(result)} 筆")
    return result

def median_vs_angle(df, angle_col, value_col):
    """
    Bin values by integer angle 0..359 and return median in each bin.
    """
    ang = np.mod(df[angle_col].to_numpy(), 360.0)
    val = df[value_col].to_numpy()

    bins = np.arange(0, 361)           # 0..360
    idx  = np.digitize(ang, bins) - 1  # -> 0..359

    out = np.full(360, np.nan)
    for deg in range(360):
        m = (idx == deg)
        if np.any(m):
            out[deg] = np.nanmedian(val[m])

    return np.arange(360), out


def moving_average_circular(x, window):
    """
    Circular moving-average smoothing for 1D data with NaNs.
    - x: array length 360 (angle-based)
    - window: window size in samples (degrees), should be >=1 (odd recommended)
    """
    x = np.asarray(x, dtype=float)
    n = x.size
    if window <= 1 or n == 0:
        return x.copy()

    # ensure window not larger than signal
    window = int(window)
    window = max(1, min(window, n))

    # circular padding
    half = window // 2
    pad = np.concatenate([x[-half:], x, x[:half]])

    # handle NaNs: weighted average ignoring NaNs
    valid = ~np.isnan(pad)
    filled = np.where(valid, pad, 0.0)

    kernel = np.ones(window, dtype=float)
    num = np.convolve(filled, kernel, mode='same')
    den = np.convolve(valid.astype(float), kernel, mode='same')
    smoothed = num / np.maximum(den, 1e-9)

    # extract center part corresponding to original
    smoothed_center = smoothed[half:half+n]
    return smoothed_center


## 3. Torque 力矩分析

此段會：

1. 讀取 `C.xlsx`。
2. 刪除前後各 60 秒。
3. 計算每個角度的中位數（Median）左右腳力矩。
4. 先平滑，再尋找「右腳由下往上超越左腳」的交叉點。
5. 把交叉點校正到約 2.5°，讓不同資料較容易對齊。
6. 再計算校正後 0–359° 的力矩曲線。
7. 將 0° 複製到 360°，讓圖形首尾閉合。


In [ ]:
print("=== Loading torque data ===")
df_tq = pd.read_excel(CPM_FILE, sheet_name=CPM_SHEET)
df_tq.columns = df_tq.columns.str.strip()
df_tq = trim_edges(df_tq, TIME_TQ, "Torque")
print("Torque columns:", df_tq.columns.tolist())

# 🌟 終極完美對齊：尋找左右腳「黃金交叉點」並閉合 360 度 🌟
print("\n--- 正在自動分析並對齊完美的倒8字型交界點 ---")


In [ ]:
_, raw_torque_R = median_vs_angle(df_tq, ANGLE_TQ, TORQUE_R)
_, raw_torque_L = median_vs_angle(df_tq, ANGLE_TQ, TORQUE_L)


In [ ]:
raw_tq_R_sm = moving_average_circular(raw_torque_R, 21)
raw_tq_L_sm = moving_average_circular(raw_torque_L, 21)


In [ ]:
# 將陣列延長兩倍，用來完美處理 0 度與 360 度的邊界搜尋
ext_R = np.concatenate([raw_tq_R_sm, raw_tq_R_sm])
ext_L = np.concatenate([raw_tq_L_sm, raw_tq_L_sm])

peak_idx = np.nanargmax(raw_tq_R_sm)
search_start = peak_idx + 360

start_idx = search_start
for i in range(search_start, search_start - 180, -1):
    # 條件：此刻右腳數值大於等於左腳，且前一刻右腳小於左腳 (代表右腳往上穿過左腳)
    if ext_R[i] >= ext_L[i] and ext_R[i-1] < ext_L[i-1]:
        start_idx = i
        break
else:
    # 備用方案：若無交叉點，則退回尋找右腳過0的點
    for i in range(search_start, search_start - 180, -1):
        if ext_R[i] >= 0 and ext_R[i-1] < 0:
            start_idx = i
            break

raw_start_deg = start_idx % 360


In [ ]:
target_start_angle = 2.5
ANGLE_OFFSET_DEG = round(target_start_angle - raw_start_deg)

print(f"🎯 偵測到左右腳的原始交叉點落在：{raw_start_deg:.1f}°")
print(f"🎯 系統已自動將 ANGLE_OFFSET_DEG 校準為：{ANGLE_OFFSET_DEG} 度")
print("---------------------------------------------------\n")

# 套用偏移量與計算最終資料
df_tq["Angle_fixed"] = np.mod(df_tq[ANGLE_TQ] + ANGLE_OFFSET_DEG, 360.0)

ang_deg, torque_median_L_raw = median_vs_angle(df_tq, "Angle_fixed", TORQUE_L)
_,       torque_median_R_raw = median_vs_angle(df_tq, "Angle_fixed", TORQUE_R)

if APPLY_TORQUE_SMOOTHING:
    torque_median_L_smooth = moving_average_circular(torque_median_L_raw, TORQUE_SMOOTH_WINDOW_DEG)
    torque_median_R_smooth = moving_average_circular(torque_median_R_raw, TORQUE_SMOOTH_WINDOW_DEG)
else:
    torque_median_L_smooth = torque_median_L_raw.copy()
    torque_median_R_smooth = torque_median_R_raw.copy()

torque_median_L = torque_median_L_smooth
torque_median_R = torque_median_R_smooth

# 🌟 繪圖優化：為了解決 360 度完美閉合，將第 0 度的資料複製到第 360 度
# 建立用來畫圖的陣列 (長度變成 361，包含 0~360)
plot_ang = np.append(ang_deg, 360)
plot_tq_L_sm = np.append(torque_median_L_smooth, torque_median_L_smooth[0])
plot_tq_R_sm = np.append(torque_median_R_smooth, torque_median_R_smooth[0])


# --------- PLOT: 完美倒8字型 (左右腳合併比較) ----------
plt.figure(figsize=(10,4))
plt.plot(plot_ang, plot_tq_R_sm, label="Right torque (smoothed)", color='orange', linewidth=2)
plt.plot(plot_ang, plot_tq_L_sm, label="Left torque (smoothed)", color='blue', linewidth=2)

# 強制 X 軸顯示 0~360，讓線條貼齊左右兩側
plt.xlim(0, 360)
plt.axhline(0, color='gray', linewidth=1, linestyle='--') # 補上一條 0 的水平虛線方便觀察

plt.xlabel("Crank angle (deg)")
plt.ylabel("Median torque")
plt.title(f"Perfect Figure-8: Left vs Right Torque (Offset = {ANGLE_OFFSET_DEG}°)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


## 4. EMG 讀取與踩踏週期切分

利用左右 Vastus Medialis（股內側肌）EMG：

- 計算 `L_VM - R_VM`。
- 找出訊號從負值穿越到正值的時間點，也就是 L > R 的交叉點。
- 兩個連續交叉點之間定義為一個完整 EMG 週期。

這裡的週期是依 EMG 交叉點定義，不代表直接取得了腳踏車曲柄角度。


In [ ]:
print("\n=== Loading EMG data ===")
df_emg = pd.read_excel(EMG_FILE, sheet_name=EMG_SHEET)
df_emg.columns = df_emg.columns.str.strip()
df_emg = trim_edges(df_emg, TIME_EMG, "EMG")
print("EMG columns:", df_emg.columns.tolist())
print(df_emg.head())

t_emg = df_emg[TIME_EMG].values
emg_cols = [c for c in df_emg.columns if c != TIME_EMG]

rVM = df_emg[R_VM_COL].values
lVM = df_emg[L_VM_COL].values

# Difference signal (L - R)
diff = lVM - rVM

# Handle zeros in sign
sign = np.sign(diff)
for i in range(1, len(sign)):
    if sign[i] == 0:
        sign[i] = sign[i-1]

# Detect L>R crossings (negative -> positive)
cross_up_times = []
for i in range(1, len(diff)):
    if sign[i-1] < 0 and sign[i] > 0:
        t0, t1 = t_emg[i-1], t_emg[i]
        y0, y1 = diff[i-1], diff[i]
        if y1 != y0:
            t_cross = t0 + (0 - y0) * (t1 - t0) / (y1 - y0)
        else:
            t_cross = 0.5 * (t0 + t1)
        cross_up_times.append(t_cross)

print(f"Number of L>R crossings found: {len(cross_up_times)}")

if len(cross_up_times) < 2:
    raise RuntimeError("Not enough L>R crossings to define at least one cycle.")

# Full cycles = between successive L>R crossings
cycle_bounds = [(cross_up_times[i], cross_up_times[i+1])
                for i in range(len(cross_up_times)-1)]
n_cycles = len(cycle_bounds)
print(f"Number of EMG cycles (L>R -> L>R): {n_cycles}")

# Optional cycle timing table
cycle_info = []
for idx, (ts, te) in enumerate(cycle_bounds, start=1):
    cycle_info.append({
        "cycle": idx,
        "t_start_s": ts,
        "t_end_s": te,
        "duration_s": te - ts
    })
cycle_info_df = pd.DataFrame(cycle_info)
print(cycle_info_df)


## 5. EMG 週期正規化為 0–359°

每一個 EMG 週期長度可能不同，因此先將每個週期換算成 0–1 的相位，再用插值重新取樣成 360 個點。

這樣：
- 第 0 點 ≈ 0°
- 第 180 點 ≈ 半個週期
- 第 359 點 ≈ 一個週期結束

接著才對所有有效週期計算平均值或中位數。


In [ ]:
print("\n=== Resampling EMG to angle domain ===")

NANG = 360                       # number of angle samples (0..359 deg)
phase_grid = np.linspace(0, 1, NANG)  # normalized phase 0..1

n_muscles = len(emg_cols)
cycles_EMG = np.full((n_cycles, NANG, n_muscles), np.nan)

for c_idx, (ts, te) in enumerate(cycle_bounds):
    mask = (t_emg >= ts) & (t_emg <= te)
    t_cycle = t_emg[mask]

    if len(t_cycle) < 2:
        print(f"Warning: cycle {c_idx+1} has too few EMG points, skipping.")
        continue

    phase = (t_cycle - ts) / (te - ts)

    for m_idx, col in enumerate(emg_cols):
        y = df_emg.loc[mask, col].values
        cycles_EMG[c_idx, :, m_idx] = np.interp(phase_grid, phase, y)

print("cycles_EMG shape (cycles, angle_samples, muscles):", cycles_EMG.shape)

# Median EMG profile per muscle across cycles
median_EMG = np.nanmedian(cycles_EMG, axis=0)   # shape: (NANG, n_muscles)


## 6. 匯出 Excel

輸出的 Excel 會包含：

- Torque 統計結果（Raw + Smoothed）
- EMG 統計結果
- 每個 EMG cycle 的開始時間、結束時間與週期長度

檔名會自動加上執行時間，避免覆蓋前一次結果。


In [ ]:
print("\n=== Exporting to Excel ===")
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_file = f"Torque_EMG_Median_Trim60s_{timestamp}.xlsx"

with pd.ExcelWriter(output_file) as writer:
    # Sheet: median torque (raw + smoothed)
    out_torque = pd.DataFrame({
        "angle_deg": ang_deg,
        "torque_left_raw":  torque_median_L_raw,
        "torque_right_raw": torque_median_R_raw,
        "torque_left_smooth":  torque_median_L_smooth,
        "torque_right_smooth": torque_median_R_smooth,
    })
    out_torque.to_excel(writer, sheet_name="median_torque", index=False)

    # Sheet: median EMG (angle 0..359, each muscle one column)
    out_emg = pd.DataFrame({"angle_deg": ang_deg})
    for m_idx, col in enumerate(emg_cols):
        out_emg[col] = median_EMG[:, m_idx]
    out_emg.to_excel(writer, sheet_name="median_emg", index=False)

    # Optional: cycle timing info
    cycle_info_df.to_excel(writer, sheet_name="cycle_info", index=False)

print(f"Saved combined file: {output_file}")


## 7. Torque 與 EMG 比較圖

最後繪製左右兩側：

- 平滑後 Torque
- 對應側 Vastus Medialis EMG

兩條曲線放在同一個 0–359° 軸上，方便觀察不同踩踏相位的力矩與肌肉活化變化。


In [ ]:
print("\n=== Plotting torque vs EMG (VM) ===")

# find indices of VM in emg_cols
idx_LVM = emg_cols.index(L_VM_COL)
idx_RVM = emg_cols.index(R_VM_COL)

# Left side comparison (smoothed torque)
plt.figure(figsize=(10,4))
plt.plot(ang_deg, torque_median_L_smooth, label="Left torque (smoothed)")
plt.plot(ang_deg, median_EMG[:, idx_LVM], label="Left VM EMG (median cycles)")
plt.xlabel("Crank angle (deg)")
plt.ylabel("Value")
plt.title("Left torque (smoothed) vs Left VM EMG (0–359°)")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

# Right side comparison (smoothed torque)
plt.figure(figsize=(10,4))
plt.plot(ang_deg, torque_median_R_smooth, label="Right torque (smoothed)")
plt.plot(ang_deg, median_EMG[:, idx_RVM], label="Right VM EMG (median cycles)")
plt.xlabel("Crank angle (deg)")
plt.ylabel("Value")
plt.title("Right torque (smoothed) vs Right VM EMG (0–359°)")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()
